# 02 用户流失预测模型\n\n## 分析目标\n- 基于用户历史行为特征，预测其在未来3天内是否会流失\n- 对比逻辑回归与XGBoost的预测效果\n- 识别影响流失的关键特征，为精细化运营提供依据\n\n## 流失定义\n- 若用户在数据集最后3天（2017-12-01 至 2017-12-03）没有任何行为记录，则标记为流失（1），否则为未流失（0）。

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport warnings\nimport os\nfrom datetime import datetime, timedelta\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, precision_score, recall_score,\n                             f1_score, roc_auc_score, roc_curve, classification_report)\nimport xgboost as xgb\n\nwarnings.filterwarnings('ignore')\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']\nplt.rcParams['axes.unicode_minus'] = False\nsns.set_style('whitegrid')\n\nIMG_DIR = '../images'\nos.makedirs(IMG_DIR, exist_ok=True)\n\nprint('环境初始化完成')

## 1. 数据加载

In [ ]:
DATA_PATH = '../data/processed/user_behavior_cleaned.csv'\n\nif os.path.exists(DATA_PATH):\n    df = pd.read_csv(DATA_PATH, parse_dates=['datetime', 'date'])\n    print('已加载真实数据')\nelse:\n    np.random.seed(42)\n    n = 100000\n    users = np.random.choice(range(1000, 9000), n)\n    items = np.random.choice(range(100000, 900000), n)\n    cates = np.random.choice(range(100, 1200), n)\n    behaviors = np.random.choice(['pv', 'buy', 'cart', 'fav'], n, p=[0.70, 0.05, 0.15, 0.10])\n    start = datetime(2017, 11, 25)\n    deltas = np.random.randint(0, 9*24*3600, n)\n    datetimes = [start + timedelta(seconds=int(s)) for s in deltas]\n    df = pd.DataFrame({\n        'user_id': users, 'item_id': items, 'category_id': cates,\n        'behavior_type': behaviors, 'timestamp': [int(d.timestamp()) for d in datetimes],\n        'datetime': datetimes,\n    })\n    df['date'] = df['datetime'].dt.date\n    df['hour'] = df['datetime'].dt.hour\n    df['day_of_week'] = df['datetime'].dt.dayofweek\n    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)\n    def time_period(h):\n        if 0 <= h < 6: return '凌晨'\n        elif 6 <= h < 12: return '上午'\n        elif 12 <= h < 18: return '下午'\n        else: return '晚上'\n    df['time_period'] = df['hour'].apply(time_period)\n    print('已生成模拟数据')\n\ndf.head()

## 2. 特征工程\n\n为每个用户构建以下统计特征：\n- 总点击次数、购买次数、加购次数、收藏次数\n- 活跃天数、活跃小时数\n- 最近一次行为距今天数（Recency）\n- 购买转化率（购买/点击）\n- 周末活跃占比

In [ ]:
# 确保date为日期类型\ndf['date'] = pd.to_datetime(df['date']).dt.date\nmax_date = df['date'].max()\nprint('最大日期:', max_date)\n\n# 用户行为统计\nuser_stats = df.groupby('user_id').agg(\n    total_pv=('behavior_type', lambda x: (x == 'pv').sum()),\n    total_buy=('behavior_type', lambda x: (x == 'buy').sum()),\n    total_cart=('behavior_type', lambda x: (x == 'cart').sum()),\n    total_fav=('behavior_type', lambda x: (x == 'fav').sum()),\n    active_days=('date', 'nunique'),\n    active_hours=('hour', 'nunique'),\n    last_date=('date', 'max'),\n    weekend_ratio=('is_weekend', 'mean'),\n).reset_index()\n\nuser_stats['recency_days'] = (max_date - user_stats['last_date']).dt.days\nuser_stats['buy_conversion'] = user_stats['total_buy'] / (user_stats['total_pv'] + 1e-6)\nuser_stats['cart_conversion'] = user_stats['total_cart'] / (user_stats['total_pv'] + 1e-6)\nuser_stats['fav_conversion'] = user_stats['total_fav'] / (user_stats['total_pv'] + 1e-6)\n\n# 流失标签：最后3天无行为\nchurn_threshold_date = max_date - timedelta(days=2)  # 最后3天：max_date-2, max_date-1, max_date\nuser_stats['churn'] = (user_stats['last_date'] < churn_threshold_date).astype(int)\n\nprint('特征样例:')\nprint(user_stats.head())\nprint('\\n流失比例:', user_stats['churn'].mean() * 100, '%')

## 3. 模型训练\n\n划分训练集与测试集（8:2），对数值特征做标准化。

In [ ]:
feature_cols = ['total_pv', 'total_buy', 'total_cart', 'total_fav',\n                'active_days', 'active_hours', 'recency_days',\n                'weekend_ratio', 'buy_conversion', 'cart_conversion', 'fav_conversion']\nX = user_stats[feature_cols]\ny = user_stats['churn']\n\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n\nprint('训练集大小:', X_train.shape, '测试集大小:', X_test.shape)

### 3.1 逻辑回归

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)\nlr.fit(X_train_scaled, y_train)\ny_pred_lr = lr.predict(X_test_scaled)\ny_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]\n\nprint('=== 逻辑回归 ===')\nprint('准确率:', accuracy_score(y_test, y_pred_lr))\nprint('精确率:', precision_score(y_test, y_pred_lr))\nprint('召回率:', recall_score(y_test, y_pred_lr))\nprint('F1分数:', f1_score(y_test, y_pred_lr))\nprint('AUC:', roc_auc_score(y_test, y_prob_lr))

### 3.2 XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(\n    n_estimators=200,\n    max_depth=5,\n    learning_rate=0.1,\n    subsample=0.8,\n    colsample_bytree=0.8,\n    random_state=42,\n    use_label_encoder=False,\n    eval_metric='logloss'\n)\nxgb_model.fit(X_train, y_train)\ny_pred_xgb = xgb_model.predict(X_test)\ny_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]\n\nprint('=== XGBoost ===')\nprint('准确率:', accuracy_score(y_test, y_pred_xgb))\nprint('精确率:', precision_score(y_test, y_pred_xgb))\nprint('召回率:', recall_score(y_test, y_pred_xgb))\nprint('F1分数:', f1_score(y_test, y_pred_xgb))\nprint('AUC:', roc_auc_score(y_test, y_prob_xgb))

## 4. 模型评估：ROC曲线对比

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)\nfpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)\n\nplt.figure(figsize=(8, 6))\nplt.plot(fpr_lr, tpr_lr, label=f'逻辑回归 (AUC={roc_auc_score(y_test, y_prob_lr):.3f})')\nplt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={roc_auc_score(y_test, y_prob_xgb):.3f})')\nplt.plot([0, 1], [0, 1], 'k--', label='随机猜测')\nplt.xlabel('假正例率 (FPR)')\nplt.ylabel('真正例率 (TPR)')\nplt.title('ROC曲线对比')\nplt.legend(loc='lower right')\nplt.tight_layout()\nplt.savefig(f'{IMG_DIR}/02_roc_curve.png', dpi=150, bbox_inches='tight')\nplt.show()

## 5. 特征重要性分析

In [ ]:
# XGBoost特征重要性\nimportance = pd.DataFrame({\n    'feature': feature_cols,\n    'importance': xgb_model.feature_importances_\n}).sort_values('importance', ascending=False)\n\nplt.figure(figsize=(10, 6))\nsns.barplot(data=importance, x='importance', y='feature', palette='viridis')\nplt.title('XGBoost 特征重要性')\nplt.xlabel('重要性')\nplt.ylabel('特征')\nplt.tight_layout()\nplt.savefig(f'{IMG_DIR}/02_feature_importance.png', dpi=150, bbox_inches='tight')\nplt.show()

## 6. 业务建议\n\n1. **Recency（最近一次行为时间）** 通常是预测流失的最强信号：用户越久未活跃，流失概率越高。建议对沉默3天以上的用户触发Push或短信召回。\n2. **活跃天数与小时数** 反映用户粘性：低活跃用户更容易流失，可通过签到、积分任务提升打开频次。\n3. **购买转化率** 低但近期有行为的用户，属于“高意向未成交”群体，适合发放定向优惠券促成首单。\n4. **模型选择**：若追求可解释性，优先使用逻辑回归；若追求预测精度，XGBoost更优。实际部署时可采用XGBoost打分+规则引擎的组合策略。